In [54]:
import cvxpy as cp
import numpy as np
import sys

In [51]:
def select_jindex(cbar, method):
    '''
    selects/outputs the j (index) that is to be entered in the simplex
    Args:
        cbar (list[float]): the reduced cost function
        method (int): the method by which to decide for the j index (either 1 or 2)

    Returns:
        output (int): the required index

    Constraints:
        0 <= output < len(cbar)
    '''
    l = len(cbar)
    if(method == 1): # bland's pivoting rule
        for i in range(l):
            if(cbar[i]<0):
                return i
    elif(method == 2): # most negative reduced cost index
        val = 0
        for i in range(l):
            if(cbar[i] <= val):
                answer = i
                val = cbar[i]
        return answer
    else:
        raise Exception("Only 2 choices for methods!")

In [100]:
def simplex(x, Basis, c, A, method, max_iters = 20):
    '''
    Runs simplex algorithm and generates the optimal solution (-1 if unbounded)
    Args:
        x (vector(float)): initialization point
        Basis (list[int]): initial basis
        c (vector(float)): cost vector
        A (matrix(float)): A of Ax = b
        method (int): which method to use to determine the j index (either 1 or 2)
        max_iters (int, default = 20): maximum number of iterations to run

    Returns:
        optimal point for the given cost function (objective = min c.T@x) or -1 if unbounded 
    '''
    B = Basis.copy()
    l = len(x)
    output = -1
    for itr in range(max_iters):
        print("Iteration:", itr)
        print("x:", x)
        print("Basis:", B)
        print("cTx (objective value):", c.T@x)
        Ab = A[:, B]
        print("AB:", Ab)
        cb = c[B]
        p = np.linalg.inv(Ab.T)@cb
        print("p:", p)
        cbar = c - A.T@p
        print("cbar:", cbar)
        if(np.all(cbar>=0)):
            output = x
            break
        d = np.zeros(l)
        jindex = select_jindex(cbar, method)
        print("j:", jindex)
        d[jindex] = 1
        d[B] = -np.linalg.inv(Ab)@A[:, jindex]
        print("dB:", d[B])
        print("d:", d)
        if(np.all(d[B]>=0)):
            break
        theta = sys.maxsize
        for i in B:
        # While computing theta, we only use i from basis (because for other i, x[i] is already 0)
        # And we use the smallest i if there is a tie
        # So, degeneracy is about this (selecting i) and not about selecting j? Also about j
        #? What does negative cost index mean here? Here, there is no cost right?
        #? Also, first rule (bland's): we select smallest i only if there is a tie right?
            if(d[i]<0):
                val = -x[i]/d[i]
                if(val<theta):
                    iindex = i
                    theta = val
        print("i:", iindex)
        print("theta*:", theta)
        x = x + d*theta
        B.append(jindex)
        B.remove(iindex)
        B = sorted(B)
        print("")
    return output

#### (a) Simplex using bland's pivoting rule: Method 1

In [109]:
# x1, x2, x3, x4, x5, x6, x7
# x1 + x4 = 2
# x2 + x5 = 2
# x3 + x6 = 2
# x1 + x2 + x3 + x7 = 4
x = np.array([2, 2, 0, 0, 0, 2, 0])
B = [0, 1, 2, 5]
c = np.array([1, 1, -1, 0, 0, 0, 0])
A = np.array([[1, 0, 0, 1, 0, 0, 0], 
              [0, 1, 0, 0, 1, 0, 0], 
              [0, 0, 1, 0, 0, 1, 0], 
              [1, 1, 1, 0, 0, 0, 1]])
b = np.array([2, 2, 2, 4])
output = simplex(x, B, c, A, 1)

Iteration: 0
x: [2 2 0 0 0 2 0]
Basis: [0, 1, 2, 5]
cTx (objective value): 4
AB: [[1 0 0 0]
 [0 1 0 0]
 [0 0 1 1]
 [1 1 1 0]]
p: [ 2.  2.  0. -1.]
cbar: [ 0.  0.  0. -2. -2.  0.  1.]
j: 3
dB: [-1. -0.  1. -1.]
d: [-1. -0.  1.  1.  0. -1.  0.]
i: 0
theta*: 2.0

Iteration: 1
x: [0. 2. 2. 2. 0. 0. 0.]
Basis: [1, 2, 3, 5]
cTx (objective value): 0.0
AB: [[0 0 1 0]
 [1 0 0 0]
 [0 1 0 1]
 [1 1 0 0]]
p: [ 0.  2.  0. -1.]
cbar: [ 2.  0.  0.  0. -2.  0.  1.]
j: 4
dB: [-1.  1. -0. -1.]
d: [ 0. -1.  1. -0.  1. -1.  0.]
i: 5
theta*: 0.0

Iteration: 2
x: [0. 2. 2. 2. 0. 0. 0.]
Basis: [1, 2, 3, 4]
cTx (objective value): 0.0
AB: [[0 0 1 0]
 [1 0 0 1]
 [0 1 0 0]
 [1 1 0 0]]
p: [ 0.  0. -2.  1.]
cbar: [ 0.  0.  0.  0.  0.  2. -1.]
j: 6
dB: [-1. -0. -0.  1.]
d: [ 0. -1. -0. -0.  1.  0.  1.]
i: 1
theta*: 2.0

Iteration: 3
x: [0. 0. 2. 2. 2. 0. 2.]
Basis: [2, 3, 4, 6]
cTx (objective value): -2.0
AB: [[0 1 0 0]
 [0 0 1 0]
 [1 0 0 0]
 [1 0 0 1]]
p: [ 0.  0. -1.  0.]
cbar: [1. 1. 0. 0. 0. 1. 0.]


In [110]:
print(output[:3])

[0. 0. 2.]


#### (b) Simplex using most negative reduced cost index

In [107]:
# x1, x2, x3, x4, x5, x6, x7
# x1 + x4 = 2
# x2 + x5 = 2
# x3 + x6 = 2
# x1 + x2 + x3 + x7 = 4
x = np.array([2, 2, 0, 0, 0, 2, 0])
B = [0, 1, 2, 5]
c = np.array([1, 1, -1, 0, 0, 0, 0])
A = np.array([[1, 0, 0, 1, 0, 0, 0], 
              [0, 1, 0, 0, 1, 0, 0], 
              [0, 0, 1, 0, 0, 1, 0], 
              [1, 1, 1, 0, 0, 0, 1]])
b = np.array([2, 2, 2, 4])
output = simplex(x, B, c, A, 2)

Iteration: 0
x: [2 2 0 0 0 2 0]
Basis: [0, 1, 2, 5]
cTx (objective value): 4
AB: [[1 0 0 0]
 [0 1 0 0]
 [0 0 1 1]
 [1 1 1 0]]
p: [ 2.  2.  0. -1.]
cbar: [ 0.  0.  0. -2. -2.  0.  1.]
j: 4
dB: [-0. -1.  1. -1.]
d: [-0. -1.  1.  0.  1. -1.  0.]
i: 1
theta*: 2.0

Iteration: 1
x: [2. 0. 2. 0. 2. 0. 0.]
Basis: [0, 2, 4, 5]
cTx (objective value): 0.0
AB: [[1 0 0 0]
 [0 0 1 0]
 [0 1 0 1]
 [1 1 0 0]]
p: [ 2.  0.  0. -1.]
cbar: [ 0.  2.  0. -2.  0.  0.  1.]
j: 3
dB: [-1.  1. -0. -1.]
d: [-1.  0.  1.  1. -0. -1.  0.]
i: 5
theta*: 0.0

Iteration: 2
x: [2. 0. 2. 0. 2. 0. 0.]
Basis: [0, 2, 3, 4]
cTx (objective value): 0.0
AB: [[1 0 1 0]
 [0 0 0 1]
 [0 1 0 0]
 [1 1 0 0]]
p: [ 0.  0. -2.  1.]
cbar: [ 0.  0.  0.  0.  0.  2. -1.]
j: 6
dB: [-1. -0.  1. -0.]
d: [-1.  0. -0.  1. -0.  0.  1.]
i: 0
theta*: 2.0

Iteration: 3
x: [0. 0. 2. 2. 2. 0. 2.]
Basis: [2, 3, 4, 6]
cTx (objective value): -2.0
AB: [[0 1 0 0]
 [0 0 1 0]
 [1 0 0 0]
 [1 0 0 1]]
p: [ 0.  0. -1.  0.]
cbar: [1. 1. 0. 0. 0. 1. 0.]


In [108]:
print(output[:3])

[0. 0. 2.]


#### Verifying solution using CVXPY

In [55]:
# To analyze using CVXPY

x = cp.Variable(3)
b = np.array([0, 0, 0, 2, 2, 2, 4])
A = np.array([[-1, 0, 0], [0, -1, 0], [0, 0, -1], [1, 0, 0], [0, 1, 0], [0, 0, 1], [1, 1, 1]])
c = np.array([1, 1, -1])

objective = cp.Minimize(c.T@x)

constraints = [A@x<=b]

problem = cp.Problem(objective, constraints)

problem.solve()

np.float64(-1.999999999530042)

In [58]:
np.round(x.value, 2)

array([-0., -0.,  2.])

##### As seen, output from each of the 2 methods (a, b) match with that of the CVXPY.